In [ ]:
# print("123")

# HOMEWORK WEEK-4

In [1]:
!pip install gitsource

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gitsource]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
import json
import pandas as pd
from evaluation_utils import llm_structured
from gitsource import GithubRepositoryDataReader, chunk_documents
from embedder import Embedder
from minsearch import Index, VectorSearch
from pydantic import BaseModel
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [ ]:
model='gemini-3.5-flash'

In [ ]:
embed_model = "models/Xenova/all-MiniLM-L6-v2"

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

In [ ]:
documents = [file.parse() for file in reader.read()]

In [ ]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [ ]:
user_prompt = """
FILENAME: {filename}

LESSON CONTENT:
{content}
""".strip()

In [ ]:
class Question(BaseModel):
    questions: list[str]

In [ ]:
messages = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(text=user_prompt), 
            types.Part.from_text(text=data_gen_instructions)
        ]
    )
]

config = types.GenerateContentConfig(
    temperature=0.2,
    max_output_tokens=2000,
    response_mime_type="application/json",  # Forces Gemini to output strict JSON
    response_schema=Question                # Enforces your structural blueprint
)

response = client.models.generate_content(
    model=model,
    contents=messages,
    config=config
)

## QUESTION 1

In [ ]:
input_tokens = []

In [ ]:
# usage = response.usage_metadata

In [ ]:
for doc in documents[:3]:
    user_prompt_template = user_prompt.format(
        filename=doc["filename"],
        content=doc["content"],
    )

    parsed, usage = llm_structured(client, data_gen_instructions, user_prompt_template, Question)

    print(doc["filename"], "->", usage.prompt_token_count, "input tokens")
    input_tokens.append(response.usage_metadata)

average_input_tokens = sum(input_tokens) / len(input_tokens)
print("\nAverage input tokens:", average_input_tokens)

## QUESTION 2 and 3

In [ ]:
ground_truth = pd.read_csv("ground-truth.csv")
first_question = ground_truth.iloc[0]["question"]


In [ ]:
chunks = chunk_documents(documents, size=2000, step=1000)

In [ ]:
text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

In [ ]:
def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

In [ ]:
text_results = text_search(first_question, num_results=5)
print("Q2 - text_search top result:", text_results[0]["filename"])

In [ ]:
embedder = Embedder(path=embed_model)

In [ ]:
print(f"Embedding {len(chunks)} chunks...")
chunk_texts = [chunk["content"] for chunk in chunks]
chunk_vectors = embedder.encode_batch(chunk_texts)   # one batched call instead of 295 API calls

In [ ]:
vector_index = VectorSearch(keyword_fields=["filename"])
vector_index.fit(chunk_vectors, chunks)

In [ ]:
def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vector_index.search(query_vector, num_results=num_results)

In [ ]:
print("\nQuestion:", first_question)

In [ ]:
vector_results = vector_search(first_question, num_results=5)
print("Q3 - vector_search top result:", vector_results[0]["filename"])

## QUESTION 4

In [ ]:
def compute_relevance(results, expected_filename):
    return [1 if r["filename"] == expected_filename else 0 for r in results]

In [ ]:
def evaluate(ground_truth, search_function):
    relevance_total = []
    for _, row in ground_truth.iterrows():
        results = search_function(row["question"])
        relevance_total.append(compute_relevance(results, row["filename"]))
    return relevance_total

In [ ]:
def hit_rate(relevance_total):
    return sum(1 for line in relevance_total if 1 in line) / len(relevance_total)

In [ ]:
ground_truth = pd.read_csv("ground-truth.csv")

In [ ]:
relevance = evaluate(ground_truth, text_search)
print(f"Hit Rate for text_search across all questions: {hit_rate(relevance)}")

## QUESTION 5

In [ ]:
def mrr(relevance_total):
    total = 0.0
    for line in relevance_total:
        for rank, is_relevant in enumerate(line, start=1):
            if is_relevant:
                total += 1 / rank
                break
    return total / len(relevance_total)

In [ ]:
print(f"MRR for vector_search across all questions: {mrr(relevance)}")

## QUESTION 6

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [ ]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)     # pull a bit deeper than final top-5
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [ ]:
for k in [1, 50, 100, 200]:
    def search_fn(query, k=k):
        return hybrid_search(query, k=k)
    relevance = evaluate(ground_truth, search_fn)
    print(f"Hybrid search: tuning the RRF parameter k: {k} , mrr: {mrr(relevance)}")